# Search → Result → Resource

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/u-kitazawa/rhinestone/blob/develop/showcase/01_search_and_resource.ipynb)

組み込みの国土地理院（GSI）定義を使い、外部通信なしでRhinestoneの基本フローを確認します。

In [1]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "git+https://github.com/u-kitazawa/rhinestone.git@fb673a0a0644333e2e4c0aad002973f817876d29",
        ],
        check=True,
    )

## 1. 検索する

対象Providerを`GSI`だけに固定します。結果順はProviderをまたぐ関連度順ではなく、構成順です。

In [2]:
from rhinestone import configure, sources
from rhinestone.models import SearchQuery

app = configure(sources=(sources.GSI,))
results = app.search(SearchQuery(text="標準地図", limit=1))

print("Providers:", results.keys())
print("Results:", len(results))
print("Diagnostics:", results.diagnostics)

Providers: ('gsi',)
Results: 1
Diagnostics: ()


## 2. Resultを調べる

`Result`は表示用メタデータ、発見元、通常の解決パイプラインへ渡すtargetを保持します。

In [3]:
result = results[0]
print("Title:", result.title)
print("Discovered by:", result.discovered_by)
print("Target:", result.target.source_id, dict(result.target.settings))

Title: 標準地図
Discovered by: gsi
Target: gsi {'id': 'std'}


## 3. Resourceへ解決する

解決後はURI、形式、来歴、実行方法を表すAccessPlanが確定します。ここではデータ本体を開きません。

In [4]:
resource = app.resolve(result)
print("URI:", resource.uri)
print("Format:", resource.format, f"({resource.media_type})")
print(
    "Provider / dataset:",
    resource.provenance.provider,
    "/",
    resource.provenance.dataset_identifier,
)
print(
    "AccessPlan:",
    type(resource.access_plan).__name__,
    f"({resource.access_plan.kind})",
)
print("Usage:", resource.metadata.raw["usage_url"])

URI: https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png
Format: png (image/png)
Provider / dataset: gsi / std
AccessPlan: RemoteDatasetPlan (remote-dataset)
Usage: https://maps.gsi.go.jp/development/ichiran.html#std
